# 05 — CNN Classification
Train a Convolutional Neural Network on the K-Means cluster labels.
Evaluate with both random and spatial (geographic) splits. Save trained artifacts.

## Train / Test Split

In [ ]:
X = reduced_data
y = labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
# Reshape for CNN input: (pixels, 10, 1, 1)
X_train_cnn = X_train.reshape(-1, 10, 1, 1)
X_test_cnn  = X_test.reshape(-1, 10, 1, 1)

## CNN Architecture

In [ ]:
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(10, 1, 1), padding='same'),
    MaxPooling2D((2, 1)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 1)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(5, activation='softmax')
])
cnn_model.summary()

## Training

In [ ]:
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_split=0.2)

## Evaluation — Random Split

In [ ]:
cnn_eval = cnn_model.evaluate(X_test_cnn, y_test)
print(f'CNN Accuracy (random split): {cnn_eval[1]:.4f}')

## Evaluation — Spatial Generalization
Train region: top half of orbital strip (rows 0–4669).  
Test region: bottom half (rows 4670–9339) — geographically unseen by the model.

In [ ]:
mid = hyperspectral_data.shape[0] // 2  # row 4670

bottom_pixels = reduced_data[mid * hyperspectral_data.shape[1]:]
bottom_labels = labels[mid * hyperspectral_data.shape[1]:]

bottom_cnn   = bottom_pixels.reshape(-1, 10, 1, 1)
spatial_eval = cnn_model.evaluate(bottom_cnn, bottom_labels, verbose=0)

print(f"Random split accuracy:  {cnn_eval[1]:.4f}")
print(f"Spatial generalization: {spatial_eval[1]:.4f}")
print("(0.09% drop confirms the CNN learned spectral features, not pixel position)")

## Save & Verify Trained Artifacts

In [ ]:
os.makedirs(save_dir, exist_ok=True)

cnn_model.save(f"{save_dir}/lunar_cnn.h5")
joblib.dump(pca,   f"{save_dir}/pca_transform.pkl")
joblib.dump(kmean, f"{save_dir}/kmeans.pkl")

print(f"Saved to {save_dir}/")
print(f"  lunar_cnn.h5       — trained CNN weights")
print(f"  pca_transform.pkl  — fitted PCA (10 components)")
print(f"  kmeans.pkl         — fitted K-Means (5 clusters)")

In [ ]:
from tensorflow.keras.models import load_model

cnn_loaded   = load_model(f"{save_dir}/lunar_cnn.h5")
pca_loaded   = joblib.load(f"{save_dir}/pca_transform.pkl")
kmean_loaded = joblib.load(f"{save_dir}/kmeans.pkl")

sample       = X_test_cnn[:100]
orig_preds   = np.argmax(cnn_model.predict(sample, verbose=0), axis=1)
loaded_preds = np.argmax(cnn_loaded.predict(sample, verbose=0), axis=1)
match        = np.sum(orig_preds == loaded_preds)
print(f"Prediction match: {match}/100 — {'✓ Model saved correctly' if match == 100 else '✗ Mismatch'}")